# MIMIC-IV Catalog Metadata Ingestion

Applies official MIMIC-IV table and column descriptions from `comments.json`
across the CDR:

**Bronze → Silver → Gold**

This notebook updates Unity Catalog metadata only.
It does not modify table data.

In [0]:
import json
from collections import defaultdict

CATALOG = "cdr_mimiciv"

COMMENTS_PATH = "/Workspace/Shared/RMIT_CDR_Capstone/04_Governance/comments.json"

In [0]:
with open(COMMENTS_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Loaded metadata for {len(metadata)} MIMIC-IV source tables.")

In [0]:
def escape_sql_comment(text):
    if text is None:
        return ""
    return str(text).replace("'", "''")


def table_exists(schema_name, table_name):
    return spark.catalog.tableExists(
        f"{CATALOG}.{schema_name}.{table_name}"
    )


def get_existing_column_comments(schema_name, table_name):
    rows = spark.sql(f"""
        SELECT
            column_name,
            comment
        FROM {CATALOG}.information_schema.columns
        WHERE table_schema = '{schema_name}'
          AND table_name = '{table_name}'
    """).collect()

    return {
        row["column_name"]: row["comment"]
        for row in rows
    }


def get_existing_table_comment(schema_name, table_name):
    rows = spark.sql(f"""
        SELECT comment
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{schema_name}'
          AND table_name = '{table_name}'
    """).collect()

    if not rows:
        return None

    return rows[0]["comment"]

## 1. Bronze — Official MIMIC-IV Metadata

Bronze is the source-aligned layer, so the official MIMIC-IV
descriptions are applied directly.

In [0]:
for table_name, table_meta in metadata.items():

    if not table_exists("bronze", table_name):
        continue

    full_table = f"{CATALOG}.bronze.{table_name}"

    # Table comment
    table_comment = table_meta.get("table_comment")

    if table_comment:
        spark.sql(f"""
            COMMENT ON TABLE {full_table}
            IS '{escape_sql_comment(table_comment)}'
        """)

    # Existing Bronze columns
    existing_columns = {
        field.name
        for field in spark.table(full_table).schema.fields
    }

    # Column comments
    for column_name, column_comment in table_meta.get("columns", {}).items():

        if column_name not in existing_columns:
            continue

        spark.sql(f"""
            ALTER TABLE {full_table}
            ALTER COLUMN `{column_name}`
            COMMENT '{escape_sql_comment(column_comment)}'
        """)

    print(f"BRONZE: {table_name}")

## 2. Silver — Carry Forward Source Metadata

Silver retains the meaning of source columns while adding transformation
and data-quality logic.

Official descriptions are therefore carried forward for matching
source columns.

Silver-specific columns remain untouched.

In [0]:
for table_name, table_meta in metadata.items():

    if not table_exists("silver", table_name):
        continue

    full_table = f"{CATALOG}.silver.{table_name}"

    existing_columns = {
        field.name
        for field in spark.table(full_table).schema.fields
    }

    # Add source table description only if Silver does not already have one
    existing_table_comment = get_existing_table_comment(
        "silver",
        table_name
    )

    if not existing_table_comment:

        table_comment = table_meta.get("table_comment")

        if table_comment:
            spark.sql(f"""
                COMMENT ON TABLE {full_table}
                IS '{escape_sql_comment(table_comment)}'
            """)

    # Carry official source descriptions to matching columns
    for column_name, column_comment in table_meta.get("columns", {}).items():

        if column_name not in existing_columns:
            continue

        spark.sql(f"""
            ALTER TABLE {full_table}
            ALTER COLUMN `{column_name}`
            COMMENT '{escape_sql_comment(column_comment)}'
        """)

    print(f"SILVER: {table_name}")

## 3. Build Source Column Glossary for Gold

Gold tables may combine information from several MIMIC-IV source tables.

Therefore, metadata is only propagated automatically when a source
column name has one clear documented meaning.

Ambiguous source definitions are not automatically applied.

In [0]:
column_definitions = defaultdict(set)

for table_name, table_meta in metadata.items():

    for column_name, column_comment in table_meta.get("columns", {}).items():

        if column_comment:
            column_definitions[column_name].add(column_comment.strip())


# Keep only columns with one unambiguous source definition
source_glossary = {
    column_name: next(iter(descriptions))
    for column_name, descriptions in column_definitions.items()
    if len(descriptions) == 1
}

print(
    f"{len(source_glossary)} source columns have "
    f"unambiguous metadata available for Gold."
)

## 4. Gold — Carry Forward Clinical Meaning

Gold contains analytical dimensions, facts and derived fields.

Source metadata is therefore applied only when:

- the Gold column retains the original source column name;
- the source meaning is unambiguous; and
- the Gold column does not already have its own description.

Existing Gold metadata is never overwritten.

In [0]:
gold_tables = [
    row["tableName"]
    for row in spark.sql(
        f"SHOW TABLES IN {CATALOG}.gold"
    ).collect()
]

for table_name in gold_tables:

    full_table = f"{CATALOG}.gold.{table_name}"

    existing_comments = get_existing_column_comments(
        "gold",
        table_name
    )

    updated = 0

    for column_name, current_comment in existing_comments.items():

        # Preserve Gold-specific descriptions
        if current_comment:
            continue

        # Only propagate known source semantics
        if column_name not in source_glossary:
            continue

        spark.sql(f"""
            ALTER TABLE {full_table}
            ALTER COLUMN `{column_name}`
            COMMENT '{escape_sql_comment(source_glossary[column_name])}'
        """)

        updated += 1

    print(f"GOLD: {table_name} — {updated} source descriptions added")

## Metadata Flow

```text
Official MIMIC-IV Documentation
              ↓
         comments.json
              ↓
           Bronze
      Official source semantics
              ↓
           Silver
      Source semantics retained
      + transformation metadata
              ↓
            Gold
      Source semantics retained
      + analytical metadata
              ↓
         AI/BI + Genie
```

In [0]:
print("Metadata ingestion complete.")